In [6]:
import h5py
import numpy as np
import pandas as pd
import urllib.request
import altair as alt
import tempfile
alt.data_transformers.enable('default')
alt.renderers.enable('mimetype')

RendererRegistry.enable('mimetype')

Regular imports above, loading of the file below

In [2]:
url = "https://github.com/UIUC-iSchool-DataViz/is445_data/raw/main/single_dicom.h5"
with tempfile.NamedTemporaryFile(suffix=".h5", delete=False) as tmp:
    urllib.request.urlretrieve(url, tmp.name)
    df = h5py.File(tmp.name, "r")
    scan = df["scan"][()]
    df.close()
# im not entirely sure how this import works, I got it from AI.
n_slices = scan.shape[0]
max_val  = 1747.0

### Visualization1: graph that goes through each slice in the ct scan and shows pixel intensity

This visualization is a simple graph that shows the clarity and resolution of each slice as you go from the top to bottom of the scan. To be honest, I got the ideas for each visualization from a friend in bio-medical engineering, and so while he tried to explain to me what a backprojection is and what exactly axial depth entails, I think the graph best shows which slices have a better resolution. 

because the CT scan is ordered and numbered, the encoding for this graph is quantitative; the data is meant to be displayed but also operated on to show averages, quantiles, and maximums, there's few operations to be done with it. I didn't really think about the color scheme, red dotted line for the top to show the maximum clarity in each slice, and heavy blue at the bottom to draw attention to the averages instead of the max. I thought the original graph looked very plain, so I asked AI to add the percentiles and the blue shading for the variance as well as to improve the formatting.

This graph isn't as interactive as i'd like it to be, but I do like how each slice has a point you can hover over to see the information instead of trying to line it up with the y-axis to see exact information. I think it's more interesting for the user to "zoom in" on each slice on their own as opposed to having a table displayed or a big report talking about each slice.

In [7]:
flat = scan.reshape(n_slices, -1)   # (36, 262144)
 
slice_stats = pd.DataFrame({ # create dataframe to be used
    "slice": np.arange(n_slices),
    "mean":  flat.mean(axis=1),
    "max":   flat.max(axis=1),
    "min":   flat.min(axis=1),
    "p25":   np.percentile(flat, 25, axis=1),
    "p75":   np.percentile(flat, 75, axis=1),})
# IQR shaded band (25th–75th percentile)
band = (
    alt.Chart(slice_stats)
    .mark_area(opacity=0.25, color="steelblue")
    .encode(
        x=alt.X("slice:Q",
                title="Slice Index (axial depth)",
                axis=alt.Axis(tickMinStep=1)),
        y=alt.Y("p25:Q", title="Pixel Intensity"),
        y2="p75:Q",))
 
# Mean intensity line + interactive points
mean_line = (
    alt.Chart(slice_stats)
    .mark_line(color="steelblue", strokeWidth=2.5)
    .encode(
        x="slice:Q",
        y="mean:Q",))
 
points = (
    alt.Chart(slice_stats)
    .mark_circle(size=70, color="steelblue")
    .encode(
        x="slice:Q",
        y="mean:Q",
        tooltip=[
            alt.Tooltip("slice:Q", title="Slice"),
            alt.Tooltip("mean:Q",  title="Mean intensity", format=".1f"),
            alt.Tooltip("p25:Q",   title="25th percentile", format=".1f"),
            alt.Tooltip("p75:Q",   title="75th percentile", format=".1f"),
            alt.Tooltip("max:Q",   title="Max intensity",   format=".1f"),],))
 
# Max intensity dashed line
max_line = (
    alt.Chart(slice_stats)
    .mark_line(color="tomato", strokeDash=[4, 3], strokeWidth=1.5)
    .encode(
        x="slice:Q",
        y="max:Q",
        tooltip=[
            alt.Tooltip("slice:Q", title="Slice"),
            alt.Tooltip("max:Q",   title="Max intensity", format=".1f"),],))
 
chart1 = (
    (band + mean_line + max_line + points)
    .properties(
        title=alt.Title(
            text="CT Scan — Per-Slice Intensity Profile",
            subtitle=[
                "Blue band = IQR (25th–75th percentile)",
                "Blue line = mean intensity · Red dashed = max intensity",
            ],
        ),
        width=650,
        height=300,
    )
    .interactive()
)
chart1_dict = chart1.to_dict()
chart1_dict['$schema'] = 'https://vega.github.io/schema/vega-lite/v4.17.0.json'
import json
with open('viz1_intensity_profile.json', 'w') as f:
    json.dump(chart1_dict, f)

## Visualization 2: go through each layer of the ct scan with a slider

I'm a big fan of this type of graph, it's a graph that displays a slice in grayscale, but the slider allows the user to go through each slice in order, almost giving the experience of being the ct scan and going from top to bottom.

This graph is mostly ordinal, the data itself is ordinal as no operations are really to be done with it, they're just positions with which to display the slices. However the slider and the color scheme are quantitative, since they have to be operated with in order to display the greyscale and intensity of each pixel. I went with greyscale because it looks more medical, I've never gotten a teeth x-ray in color so it would be strange to display it in blue or red or some other hue. 

The interactivity in this graph comes from the slice slider. It reminds me of something you'd see in a museum in order to look through the different phases of an animal or a geological artifact. It's not very technically impressive but it's almost always a cool experience for the user to be able to flip through the images in order to create a 3d image in their mind of what the brain scan looks like.

In [8]:
step = 8
rows = []
#create a smaller version of the dataframe since loading the giant one would make my computer catch on fire
for s in range(n_slices):
    slc = scan[s][::step, ::step]
    h, w = slc.shape
    ys, xs = np.meshgrid(np.arange(h), np.arange(w), indexing="ij")
    rows.append(
        pd.DataFrame({
            "slice":     s,
            "x":         xs.ravel(),
            "y":         ys.ravel(),
            "intensity": slc.ravel(),}))
heatmap_df = pd.concat(rows, ignore_index=True)
 
# Slider binding, make sure it starts in the center
slice_sel = alt.selection_point(
    name="slice_sel",
    fields=["slice"],
    bind=alt.binding_range(min=0, max=n_slices - 1, step=1, name="Slice:  "),
    value=18,)
 
chart2 = (
    alt.Chart(heatmap_df)
    .mark_rect()
    .encode(
        x=alt.X("x:O", axis=None),
        y=alt.Y("y:O", axis=None, sort="descending"),
        color=alt.Color(
            "intensity:Q",
            scale=alt.Scale(scheme="greys", domain=[0, max_val]),
            legend=alt.Legend(title="Intensity"),
        ),
        tooltip=[
            alt.Tooltip("slice:Q",     title="Slice"),
            alt.Tooltip("x:Q",         title="X pixel"),
            alt.Tooltip("y:Q",         title="Y pixel"),
            alt.Tooltip("intensity:Q", title="Intensity", format=".0f"),
        ],
    )
    .add_params(slice_sel)
    .transform_filter(slice_sel)
    .properties(
        title=alt.Title(
            text="CT Scan — Interactive Axial Slice Viewer",
            subtitle="Use the slider to navigate through all 36 slices (downsampled to 64×64)",
        ),
        width=512,
        height=512,))
chart2.save("viz2_slice_viewer.json")
chart2_dict = chart2.to_dict()
chart2_dict['$schema'] = 'https://vega.github.io/schema/vega-lite/v4.17.0.json'
with open('viz2_slice_viewer.json', 'w') as f:
    json.dump(chart2_dict, f)

MaxRowsError: The number of rows in your dataset (147456) is greater than the maximum allowed (5000).

Try enabling the VegaFusion data transformer which raises this limit by pre-evaluating data
transformations in Python.
    >> import altair as alt
    >> alt.data_transformers.enable("vegafusion")

Or, see https://altair-viz.github.io/user_guide/large_datasets.html for additional information
on how to plot large datasets.